In [ ]:
# installation des programmes
!pip install unidecode
!pip install selenium
!pip install ydata_profiling

In [ ]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from selenium.webdriver import ChromeOptions
from datetime import datetime
from unidecode import unidecode
from ast import literal_eval
from itertools import chain
from ydata_profiling import ProfileReport

In [ ]:
#lire le fichier csv district polynésie
df = pd.read_csv(
    '/content/districts2017.csv',
    sep=';',
    encoding='latin1',
    engine='python'
)
# Decode l'encodage pour le format latin
df['District'] = (df['District'].str.encode('latin1').str.decode('utf-8'))
df['Quartier'] = (df['Quartier'].str.encode('latin1').str.decode('utf-8'))

In [ ]:
# Paramètres par défaut pour visualiser l'ensemble des données
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)   # voir tout le contenu texte
pd.set_option('display.width', None)          # pas de retour à la ligne forcé

In [ ]:
# Nettoyage des initales et répétitions des iles au débuts de chaques adresses
df['District'] = df['District'].str.split('-', n=1).str[1]


In [ ]:
df.columns

In [ ]:
df2 = df[['District', 'Quartier', 'Commune','Ile', 'Sub']]

In [ ]:
# Dictionnaire mapping : commune → code postal
postal_map = {
    "Ahe": "98771",
    "Amaru": "98752",
    "Anaa": "98760",
    "Anatonu": "98750",
    "Arue": "98701",
    "Arutua": "98761",
    "Atuona": "98741",
    "Avatoru": "98775",
    "Bora Bora": "98730",
    "Faa'a": "98702",
    "Fakarava": "98763",
    "Fangatau": "98765",
    "Fare": "98731",
    "Fareeru": "98731",
    "Fatu Hiva":"98740",
    "Gambier": "98755",
    "Haapiti": "98729",
    "Hao":"98767",
    "Huahine": "98731",
    "Hikueru": "98768",
    "Hitiaa O Te Ra": "98705",
    "Hiva Oa": "98741",
    "Kauehi": "98763",
    "Mahanatoa": "98750",
    "Mahina": "98709",
    "Manihi": "98771",
    "Mata'ura": "98754",
    "Mataiea": "98726",
    "Maupiti": "98732",
    "Makemo": "98769",
    "Moorea-Maiao": "98728",
    "Motu Tape": "98730",
    "Motu Tautau": "98733",
    "Opoa": "98735",
    "Paea": "98711",
    "Papara": "98712",
    "Pape'ari": "98727",
    "Papeete": "98714",
    "Patio": "98733",
    "Pirae": "98716",
    "Pueu": "98721",
    "Punaauia": "98718",
    "Pukapuka": "98774",
    "Rapa": "98751",
    "Rairua": "98750",
    "Rangiroa": "98776",
    "Rikitea": "98755",
    "Raivavae": "98750",
    "Rimatara": "98752",
    "Rurutu": "98753",
    "Reao": "98779",
    "Napuka": "98772",
    "Nuku Hiva": "98742",
    "Nukutavake": "98773",
    "Tahuata": "98743",
    "Tahaa": "98733",
    "Taiarapu-Est": "98722",
    "Taiarapu-Ouest": "98723",
    "Taiohae": "98742",
    "Taravao": "98719",
    "Takaroa": "98781",
    "Taputapuatea": "98735",
    "Teahupoo": "98719",
    "Teva I Uta": "98726",
    "Tatakoto": "98783",
    "Tikehau": "98778",
    "Tumaraa": "98735",
    "Toanoano": "98725",
    "Tubuai": "98754",
    "Tureia": "98784",
    "Tuherahera": "98778",
    "Turipaoa": "98771",
    "Ua Pou": "98745",  # 98745 / 98746 → référence prise : 98745
    "Uturoa": "98735",
    "Ua Huka": "98744",
    "Vaitoare": "98734",
    "Vaiuru": "98750"
}

In [ ]:
# Selon la commune on attribue tels code postal
df2['CodePostale'] = df2["Commune"].map(postal_map)

In [ ]:
df2['Ville'] = df2[['Commune', 'Ile', 'Sub']].astype(str).agg('-'.join, axis=1)
df2["Ville"]

In [ ]:
df3 = df2[['Sub', 'Quartier', 'District', 'CodePostale']]
df3.columns = ['Ville', 'Quartier', 'Adresse', 'CodePostal']

In [ ]:
df3['Quartier'].unique()

In [ ]:
#recupérer les villes
arr = df3['Ville'].unique()
df3V = pd.DataFrame(arr, columns=['Ville'])
output_file = "df3V_export.xlsx"
df3V.to_excel(output_file, index=False)
from google.colab import files
files.download(output_file)

In [ ]:
#recupérer les quartier
arr = df3['Quartier'].unique()
df3Q = pd.DataFrame(arr, columns=['Ville'])
output_file = "df3Q_export.xlsx"
df3Q.to_excel(output_file, index=False)
from google.colab import files
files.download(output_file)

In [ ]:
# Recuperer les codes postaux
arr = df3['CodePostal'].unique()
df3CP = pd.DataFrame(arr, columns=['CodePostal'])
output_file = "df3CP_export.xlsx"
df3CP.to_excel(output_file, index=False)
from google.colab import files
files.download(output_file)

In [ ]:
output_file = "df3_export.xlsx"
df3.to_excel(output_file, index=False)
from google.colab import files
files.download(output_file)

In [ ]:
output_file = "df2_export.xlsx"
df2.to_excel(output_file, index=False)
from google.colab import files
files.download(output_file)